# **CatBoost**

CatBoost (**Categorical Boosting**) is the third giant in the gradient boosting trifecta. It was built by Yandex specifically to solve two massive issues found in XGBoost and LightGBM:

1. **Handling Categorical Data Natively:** No more manual One-Hot encoding or Target Encoding that causes data leakage.
2. **Preventing Overfitting (Prediction Shift):** Traditional boosting uses the same data points to calculate gradients and to build the tree, leading to a subtle bias. CatBoost fixes this using **Ordered Boosting**.

Before diving into the math on a tiny dataset, let's look at the two architectural pillars that make CatBoost unique.

---

## The Two Core Pillars of CatBoost

### 1. Symmetric (Oblivious) Trees

Unlike XGBoost (level-wise) or LightGBM (leaf-wise), CatBoost builds **Symmetric Trees**. This means that **at any given depth of the tree, the exact same split condition (same feature and same threshold) is applied across all nodes.**

* **Why?** It acts as a powerful regularizer preventing overfitting, and it allows for blazing-fast inference because the path to a leaf can be calculated using simple bitwise operators.

### 2. Ordered Target Encoding & Ordered Boosting

If you use standard target encoding (replacing a category like "City" with the mean of the target variable $y$), you introduce **target leakage** because the target of a row is used to encode its own feature.

CatBoost solves this by introducing a artificial "time" via **random permutations**. To encode a categorical value or calculate a gradient for a specific row, it **only looks at rows that come before it** in that random shuffling.

---

## Mathematical Application on a Tiny Dataset

Let's trace a CatBoost Regressor manually. To show how it handles categorical data without leakage, we will include a categorical feature.

### 1. The Dataset & Configurations

* **Objective:** Mean Squared Error (MSE) $\rightarrow L = \frac{1}{2}(y - \hat{y})^2$
* **Learning Rate ($\eta$):** $0.1$
* **Prior ($P$):** An initial guestimate for target statistics (commonly the average of all targets in the dataset, which is $\frac{10 + 20 + 30}{3} = 20$).
* **Weight ($w$):** A regularization weight for the prior, let's set $w = 1$.

| Row ($i$) | City (Categorical) | Target ($y$) |
| --- | --- | --- |
| 1 | Quetta | **10** |
| 2 | Lahore | **20** |
| 3 | Quetta | **30** |

To execute **Ordered Target Encoding**, CatBoost shuffles the data randomly. Let's assume the random permutation keeps the original order: `[Row 1, Row 2, Row 3]`.

---

### Step 1: Calculate Natively Encoded Numerical Values

For each row, the categorical feature value is converted to a number using **only history** (rows appearing before it in the permutation):

$$\text{Encoded Value} = \frac{\sum \text{Targets of this category in previous rows} + (w \times P)}{\text{Count of this category in previous rows} + w}$$

* **Row 1 (`City = Quetta`):** No history exists before Row 1.

$$\text{Encoded}_1 = \frac{0 + (1 \times 20)}{0 + 1} = 20$$


* **Row 2 (`City = Lahore`):** No "Lahore" history exists before Row 2.

$$\text{Encoded}_2 = \frac{0 + (1 \times 20)}{0 + 1} = 20$$


* **Row 3 (`City = Quetta`):** Row 1 was "Quetta" with a target of $10$.

$$\text{Encoded}_3 = \frac{10 + (1 \times 20)}{1 + 1} = \frac{30}{2} = 15$$



Our transformed numerical dataset for this permutation looks like this:

| Row ($i$) | Transformed Feature ($X_{encoded}$) | Target ($y$) |
| --- | --- | --- |
| 1 | 20 | 10 |
| 2 | 20 | 20 |
| 3 | 15 | 30 |

---

### Step 2: Compute Residuals (Gradients)

The base model prediction $F_0(X)$ is the global mean: $20$.
Since $L = \frac{1}{2}(y - \hat{y})^2$, the negative gradient (residual) is simply $y_i - \hat{y}_i$.

* **Residual 1:** $10 - 20 = -10$
* **Residual 2:** $20 - 20 = 0$
* **Residual 3:** $30 - 20 = 10$

---

### Step 3: Find the Best Split (Enforcing Symmetry)

We look for a threshold on $X_{encoded}$ to split our data. A clean choice between $15$ and $20$ is **Split: $X_{encoded} \le 17.5$**.

Let's check where our points fall:

* **Left Leaf ($X_{encoded} \le 17.5$):** Row 3 (Residual = $10$)
* **Right Leaf ($X_{encoded} > 17.5$):** Row 1 and Row 2 (Residuals = $-10, 0$)

---

### Step 4: Calculate Leaf Outputs

The output value $v$ of a leaf minimizes the loss of the points assigned to it. For MSE, the leaf value is the average of the residuals inside it.

* **Left Leaf Output ($v_L$):** Only contains Row 3.

$$v_L = 10$$


* **Right Leaf Output ($v_R$):** Contains Row 1 and Row 2.

$$v_R = \frac{-10 + 0}{2} = -5$$



---

### Step 5: Update Predictions

We update our predictions using our learning rate $\eta = 0.1$:


$$F_1(X) = F_0(X) + \eta \cdot v$$

* **Row 1 ($X_{encoded} = 20 \rightarrow$ Right Leaf):**

$$F_1(1) = 20 + 0.1 \times (-5) = 19.5$$


* **Row 2 ($X_{encoded} = 20 \rightarrow$ Right Leaf):**

$$F_1(2) = 20 + 0.1 \times (-5) = 19.5$$


* **Row 3 ($X_{encoded} = 15 \rightarrow$ Left Leaf):**

$$F_1(3) = 20 + 0.1 \times (10) = 21.0$$



---

## Final Intuition Comparison

CatBoost repeating this process over *multiple random permutations* of the data ensures that the model learns unbiased target statistics and structure weights.

* **XGBoost** maps out splits horizontally using all structural data points simultaneously.
* **LightGBM** dives straight down vertically, splitting single cells maximizing error drop instantly.
* **CatBoost** forces structural harmony through symmetric trees and isolates processing history through ordered steps to stay perfectly generalized.


In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# ==========================================
# 1. LOAD AND PREPROCESS MNIST DATA
# ==========================================
print("Fetching MNIST dataset (this might take a minute)...")
# fetch_openml returns images as integers 0-255; cache=True speeds up future runs
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

X = mnist.data.astype('float32')
y = mnist.target.astype('int32')

# Normalize pixel values to [0, 1] for stable training
X /= 255.0

# Split into train and test sets (using a subset if you want instant execution)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Shape: {X_train.shape}, Test Shape: {X_test.shape}\n")

# ==========================================
# 2. XGBOOST IMPLEMENTATION
# ==========================================
print("--- Training XGBoost ---")
# Use hist tree method for massive speedups similar to LightGBM
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=10,
    tree_method='hist', 
    eval_metric='mlogloss',
    learning_rate=0.1,
    max_depth=6,
    n_estimators=50,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
print(f"XGBoost Accuracy: {accuracy_score(y_test, xgb_preds):.4f}\n")


# ==========================================
# 3. LIGHTGBM IMPLEMENTATION
# ==========================================
print("--- Training LightGBM ---")
# LightGBM handles multi-class natively with leaf-wise growth
lgb_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=10,
    learning_rate=0.1,
    max_depth=-1, # Unconstrained depth, controlled by num_leaves
    num_leaves=31,
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train)
lgb_preds = lgb_model.predict(X_test)
print(f"LightGBM Accuracy: {accuracy_score(y_test, lgb_preds):.4f}\n")


# ==========================================
# 4. CATBOOST IMPLEMENTATION
# ==========================================
print("--- Training CatBoost ---")
# CatBoost builds symmetric (oblivious) trees
cb_model = cb.CatBoostClassifier(
    loss_function='MultiClass',
    iterations=50,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    verbose=10 # Prints metrics every 10 iterations
)
cb_model.fit(X_train, y_train)
cb_preds = cb_model.predict(X_test)
# CatBoost predict returns 2D array [[pred]], we flatten to 1D
cb_preds = cb_preds.flatten() 
print(f"CatBoost Accuracy: {accuracy_score(y_test, cb_preds):.4f}\n")


# ==========================================
# 5. FINAL COMPARISON REPORT
# ==========================================
print("=== FINAL PERFORMANCE METRICS (LightGBM Example) ===")
print(classification_report(y_test, lgb_preds))